In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse


class CF:
    def __init__(self, Y_data, k=30, uuCF=1):
        self.uuCF = uuCF
        self.k = k
        self.Y_data = Y_data if uuCF else Y_data[:, [1, 0, 2]]

        # Số lượng user và item
        self.n_users = int(np.max(self.Y_data[:, 0])) + 1
        self.n_items = int(np.max(self.Y_data[:, 1])) + 1

    # 1. Chuẩn hóa dữ liệu
    def normalize_Y(self):
        """
        Trừ đi trung bình rating của mỗi user
        """
        self.Ybar_data = self.Y_data.copy()
        self.mu = np.zeros(self.n_users)  

        for u in range(self.n_users):
            # Lấy tất cả rating của user u
            ids = np.where(self.Y_data[:, 0] == u)[0]

            ratings = self.Y_data[ids, 2]

            # Tính trung bình
            if len(ratings) > 0:
                mean = np.mean(ratings)
            else:
                mean = 0

            self.mu[u] = mean
            # Chuẩn hóa rating
            self.Ybar_data[ids, 2] = ratings - mean

        # Tạo sparse matrix (item x user)
        self.Ybar = sparse.coo_matrix(
            (self.Ybar_data[:, 2],
             (self.Ybar_data[:, 1], self.Ybar_data[:, 0])),
            shape=(self.n_items, self.n_users)
        ).tocsr()

    # 2. Tính similarity
    def similarity(self):
        """
        Tính độ giống nhau giữa user-user
        """
        self.S = cosine_similarity(self.Ybar.T, self.Ybar.T)

    # 3. Train model
    def fit(self):
        self.normalize_Y()
        self.similarity()

    # 4. Predict rating
    def __pred(self, u, i, normalized=1):
        """
        Dự đoán rating của user u cho item i
        """

        # Bước 1: tìm user đã rate item i
        ids = np.where(self.Y_data[:, 1] == i)[0]
        users_rated_i = self.Y_data[ids, 0].astype(int)

        # Bước 2: lấy similarity
        sim = self.S[u, users_rated_i]

        # Bước 3: chọn top-k user giống nhất
        top_k_idx = np.argsort(sim)[-self.k:]
        nearest_sim = sim[top_k_idx]

        # Bước 4: lấy rating đã normalize
        ratings = self.Ybar[i, users_rated_i[top_k_idx]].toarray().flatten()

        # Bước 5: tính weighted average
        if np.sum(np.abs(nearest_sim)) == 0:
            return 0

        pred = np.dot(ratings, nearest_sim) / (np.sum(np.abs(nearest_sim)) + 1e-8)

        # Nếu cần rating gốc thì cộng lại mean
        if normalized == 0:
            pred += self.mu[u]

        return pred

    def pred(self, u, i, normalized=1):
        if self.uuCF:
            return self.__pred(u, i, normalized)
        else:
            return self.__pred(i, u, normalized)

    # 5. Recommend
    def recommend(self, u, top_n=5):
        """
        Gợi ý top_n item cho user u
        """
        ids = np.where(self.Y_data[:, 0] == u)[0]
        rated_items = self.Y_data[ids, 1]

        predictions = []

        for i in range(self.n_items):
            if i not in rated_items:
                score = self.__pred(u, i, normalized=0)
                predictions.append((i, score))

        # Sắp xếp giảm dần
        predictions.sort(key=lambda x: x[1], reverse=True)
        return predictions[:top_n]

In [ ]:
from sklearn.model_selection import train_test_split

ratings_base = pd.read_csv('../ml-100k/ub.base', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
ratings_test = pd.read_csv('../ml-100k/ub.test', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

rate_train = ratings_base.values
rate_test = ratings_test.values

rate_train[:, :2] -= 1
rate_test[:, :2] -= 1
ratings_base.head()

,user_id,item_id,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,3,4,878542960
3,1,4,3,876893119
4,1,5,3,889751712


In [30]:
# Train model
rs = CF(rate_train, k=30, uuCF=1)
rs.fit()

n_tests = rate_test.shape[0]

SE = 0          # tổng bình phương lỗi
abs_error = 0   # tổng lỗi tuyệt đối

for i in range(n_tests):
    user = int(rate_test[i, 0])
    item = int(rate_test[i, 1])
    true_rating = rate_test[i, 2]

    pred = rs.pred(user, item, normalized=0)

    SE += (pred - true_rating) ** 2
    abs_error += abs(pred - true_rating)

RMSE = np.sqrt(SE / n_tests)
MAE = abs_error / n_tests

print("User-user CF:")
print("RMSE =", RMSE)
print("MAE =", MAE)

User-user CF:
RMSE = 1.083384587917046
MAE = 0.8249129804036938


In [31]:
rs = CF(rate_train, k = 30, uuCF = 0)
rs.fit()

n_tests = rate_test.shape[0]
SE = 0 # squared error
abs_error = 0 # absolute error
for n in range(n_tests):
    pred = rs.pred(rate_test[n, 0], rate_test[n, 1], normalized = 0)
    SE += (pred - rate_test[n, 2])**2
    abs_error += abs(pred - rate_test[n, 2]) 

RMSE = np.sqrt(SE/n_tests)
MAE = abs_error/n_tests
print('Item-item CF, RMSE =', RMSE)
print('Item-item CF, MAE =', MAE)

Item-item CF, RMSE = 1.0023177238258532
Item-item CF, MAE = 0.7908313137032572


In [33]:
def print_recommendation(self):
        """
        in ra gợi ý cho tất cả item cho user hoặc tất cả user cho item
        """
        print('Recommendation: ')
        for u in range(self.n_users):
            recommended_items = self.recommend(u)
            if self.uuCF:
                print('    Recommend item(s):', recommended_items, 'for user', u)
            else: 
                print('    Recommend item', u, 'for user(s) : ', recommended_items)

CF.print_recommendation = print_recommendation
rs.print_recommendation()

Recommendation: 
    Recommend item 0 for user(s) :  [(687, np.float64(4.835155466464585)), (506, np.float64(4.826617994932157)), (810, np.float64(4.718928873881357)), (211, np.float64(4.697117694907307)), (257, np.float64(4.687517432851642))]
    Recommend item 1 for user(s) :  [(687, np.float64(4.473901490389355)), (506, np.float64(4.054864620542533)), (354, np.float64(3.9433893710595576)), (85, np.float64(3.9396134848356192)), (848, np.float64(3.890542181345736))]
    Recommend item 2 for user(s) :  [(506, np.float64(4.070003070352159)), (241, np.float64(4.037684579027484)), (627, np.float64(3.9090284422415538)), (323, np.float64(3.7074304269909826)), (165, np.float64(3.679302617556497))]
    Recommend item 3 for user(s) :  [(506, np.float64(4.3243007748900055)), (151, np.float64(4.287154420330473)), (533, np.float64(4.247074463828492)), (259, np.float64(4.246691680942466)), (136, np.float64(4.2100246527877605))]
    Recommend item 4 for user(s) :  [(687, np.float64(4.44303679588930

KeyboardInterrupt: 